<a href="https://colab.research.google.com/github/MachineLearning30110/MachineLearning-Models/blob/main/Automated_ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [81]:
import pandas as pd
import numpy as np

np.random.seed(42)

num_entries = 1000

data = {
    'age': np.random.randint(18, 65, num_entries),
    'sex': np.random.choice(['Male', 'Female'], num_entries),
    'salary': np.random.randint(30000, 120000, num_entries),
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], num_entries),
    'target': np.random.randint(100, 300, num_entries)
}

In [82]:
df = pd.DataFrame(data)
display(df.head())

,age,sex,salary,city,target
0,56,Male,59241,Chicago,218
1,46,Female,74569,Houston,232
2,32,Female,41745,Chicago,155
3,60,Female,56029,Chicago,195
4,25,Male,43025,New York,277


### Introduce Missing Values

To simulate real-world data, I'll randomly set approximately 5% of the values in the 'Age' and 'Salary' columns to `NaN` (Not a Number).

In [83]:
# Introduce NaN values in 'Age' and 'Salary' columns
percentage_nan = 0.05 # 5% of values will be NaN

for col in ['age', 'salary']:
    mask = np.random.rand(len(df)) < percentage_nan
    df.loc[mask, col] = np.nan

print(f"Number of NaN values introduced in 'Age': {df['age'].isnull().sum()}")
print(f"Number of NaN values introduced in 'Salary': {df['salary'].isnull().sum()}")

display(df.tail())

Number of NaN values introduced in 'Age': 49
Number of NaN values introduced in 'Salary': 46


,age,sex,salary,city,target
995,22.0,Male,36256.0,Houston,137
996,40.0,Male,106256.0,Los Angeles,294
997,27.0,Female,63937.0,Houston,265
998,61.0,Male,31651.0,Los Angeles,294
999,19.0,Male,80288.0,New York,160


In [84]:
with open('/content/data.csv','w') as f:
    f.write(df.to_csv(index=False))

In [104]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [86]:
# =========================
# 1) CONFIG
# =========================
DATA_PATH = "/content/data.csv"
TARGET_COL = "target"
MODEL_DIR = "/content/models/"
BASE_MODEL_PATH = os.path.join(MODEL_DIR, "baseline_model.pkl")
TUNED_MODEL_PATH = os.path.join(MODEL_DIR, "tuned_model.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)

In [87]:
# =========================
# 2) LOAD DATA
# =========================
df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

In [88]:
# =========================
# 3) AUTO DETECT COLUMN TYPES
# =========================
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

In [89]:
# =========================
# 4) PREPROCESSING PIPELINES
# =========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [90]:
# =========================
# 5) TASK DETECTION
#    Set your own task if you know it.
# =========================
# Change this manually if needed:
TASK = "regression"   # or "classification"

In [91]:
# =========================
# 6) TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [105]:
# =========================
# 7) BASE MODEL
# =========================
if TASK == "regression":
    model = RandomForestRegressor(random_state=42)
else:
    model = RandomForestClassifier(random_state=42)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [106]:
# =========================
# 8) TRAIN BASE MODEL
# =========================
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'city'])])),
                ('model', RandomForestRegressor(random_state=42))])

In [107]:
# =========================
# 9) EVALUATE BASE MODEL
# =========================
def evaluate_model(model, X_test, y_test, task="regression"):
    y_pred = model.predict(X_test)

    if task == "regression":
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        return {"rmse": rmse, "r2": r2}
    else:
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="weighted")
        return {"accuracy": acc, "f1_weighted": f1}


base_metrics = evaluate_model(pipeline, X_test, y_test, TASK)
print("Base model metrics:", base_metrics)

Base model metrics: {'rmse': np.float64(62.700401390810484), 'r2': -0.2769108534494138}


In [108]:
# =========================
# 10) SAVE BASE MODEL
# =========================
def saveModel(model, model_path):
    joblib.dump(model, model_path)
    print(f"Saved model to: {model_path}")

saveModel(pipeline, BASE_MODEL_PATH)

Saved model to: /content/models/baseline_model.pkl


In [109]:
# =========================
# 11) TUNING FUNCTION
# =========================
def tune_model(pipeline, X_train, y_train, task="regression"):
    if task == "regression":
        param_grid = {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [None, 5, 10, 20],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
        scoring = "r2"
    else:
        param_grid = {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [None, 5, 10, 20],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
        scoring = "f1_weighted"

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring=scoring,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)
    return search

In [110]:
# =========================
# 12) TUNE IF NEEDED
# =========================
# Example rule: tune if base performance is not good enough
NEED_TUNING = True

best_model = pipeline
best_metrics = base_metrics

if NEED_TUNING:
    search = tune_model(pipeline, X_train, y_train, TASK)
    tuned_model = search.best_estimator_

    tuned_metrics = evaluate_model(tuned_model, X_test, y_test, TASK)
    print("Best params:", search.best_params_)
    print("Tuned model metrics:", tuned_metrics)

    best_model = tuned_model
    best_metrics = tuned_metrics

    # Save tuned model
    joblib.dump(best_model, TUNED_MODEL_PATH)
    print(f"Saved tuned model to: {TUNED_MODEL_PATH}")


Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best params: {'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Tuned model metrics: {'rmse': np.float64(57.72172640560805), 'r2': -0.08217764975796049}
Saved tuned model to: /content/models/tuned_model.pkl
